<a href="https://colab.research.google.com/github/terrydw-hcc/ITAI-1371-ML-Labs/blob/main/L11_TerryWilliams_ITAI1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 11 Lab - Hyperparameter Tuning & AutoML

**Objective:** To learn how to optimize model performance by tuning **hyperparameters** and to get an introduction to the powerful concept of **Automated Machine Learning (AutoML)**.

**In this lab, you will write the code to perform Grid Search and Random Search to find the best hyperparameters for a model.**

## Part 1: What are Hyperparameters?

**Concept:** In machine learning, there are two types of parameters:

1.  **Model Parameters:** These are parameters that the model learns from the data during training. For example, the coefficients in a Linear Regression model.

2.  **Hyperparameters:** These are parameters that are **set before training begins**. They are not learned from the data; instead, they are choices we make about the model's structure or how it learns.
    *   *Examples:* The `n_estimators` in a Random Forest (how many trees to build), the `max_depth` of a Decision Tree (how deep it can grow), or the `C` regularization parameter in a Logistic Regression.

Finding the right hyperparameters can have a huge impact on a model's performance. **Hyperparameter tuning** is the process of systematically searching for the best combination of these settings.

## Part 2: Setup

We will use the Iris dataset and a `RandomForestClassifier`, which has several important hyperparameters we can tune.

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load and prepare data -- standard Iris setup, 70/30 split with a fixed seed for reproducibility
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a baseline model with all DEFAULT hyperparameters.
# Per the lecture's shoe analogy (Slide 3): defaults are like buying shoes in one standard size --
# they walk safely on many datasets but aren't tuned to ours specifically. The baseline tells us
# what we have to BEAT to justify the cost of tuning.
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"Accuracy of baseline Random Forest: {accuracy_baseline:.2%}")

Accuracy of baseline Random Forest: 100.00%


## Part 3: Grid Search

**Concept:** Grid Search is the most straightforward tuning method. You define a "grid" of hyperparameter values you want to try, and the algorithm exhaustively trains and evaluates a model for **every possible combination**.

*   **Pro:** It's guaranteed to find the best combination within the grid.
*   **Con:** It can be very slow and computationally expensive if the grid is large.

### Task 1: Perform a Grid Search

**Your Task:** Use `GridSearchCV` from `sklearn.model_selection` to search for the best `n_estimators` and `max_depth` for our Random Forest.

In [2]:
from sklearn.model_selection import GridSearchCV

# 1. Define the grid of hyperparameter values to search.
# 3 values for n_estimators x 3 values for max_depth = 9 unique combinations.
# With cv=5 below, that's 9 * 5 = 45 model fits in total. Manageable.
# (The lecture's Slide 9 warning: 3 params * 10 values each = 1,000 configs, before counting CV --
# this is why grid search doesn't scale to many hyperparameters.)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None]   # None means "grow trees as deep as needed"
}

# 2. Build the GridSearchCV.
# - cv=5 makes cross-validation the referee (Slide 5) -- a fair comparison across configs
#   instead of relying on one lucky train/test split.
# - n_jobs=-1 parallelizes across all CPU cores so the search is faster.
# - verbose=2 prints progress so we can watch it work.
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)

# 3. Run the search -- fits one Random Forest per (combination, fold) pair, then ranks them
grid_search.fit(X_train, y_train)

# 4. Report the winner and its cross-validated score
print(f"\nBest Parameters found by Grid Search: {grid_search.best_params_}")
print(f"Best cross-validated score: {grid_search.best_score_:.2%}")

# Also evaluate the tuned model on the held-out test set so we can compare apples-to-apples
# against the baseline. The test set is the lecture's "holdout data" check (Slide 24).
y_pred_grid = grid_search.predict(X_test)
accuracy_grid = accuracy_score(y_test, y_pred_grid)
print(f"Test-set accuracy of the tuned model: {accuracy_grid:.2%}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits

Best Parameters found by Grid Search: {'max_depth': 5, 'n_estimators': 100}
Best cross-validated score: 94.29%
Test-set accuracy of the tuned model: 100.00%


## Part 4: Random Search

**Concept:** Random Search is often more efficient than Grid Search. Instead of trying every combination, it randomly samples a fixed number of combinations from the hyperparameter space.

*   **Pro:** It's much faster and can explore a wider range of values.
*   **Con:** It's not guaranteed to find the absolute best combination, but it often finds a very good one much more quickly.

### Task 2: Perform a Random Search

**Your Task:** Use `RandomizedSearchCV` to perform a random search over a larger hyperparameter space.

In [3]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np   # Needed for np.linspace below; not in the original imports

# 1. Define a LARGER hyperparameter space than the grid above.
# Note 'min_samples_split' is added too -- random search lets us afford a third dimension
# that grid search couldn't include without exploding the cost.
# Total possible combinations here: 10 * 5 * 3 = 150. Random search will only try 10 of them.
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start=50, stop=500, num=10)],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

# 2. Build RandomizedSearchCV.
# - n_iter=10 means it samples 10 random combinations (out of 150 possible).
# - The lecture's restaurant analogy (Slide 10): visiting every block in a city is systematic,
#   but sampling neighborhoods finds promising districts much faster. That's what's happening here.
# - random_state=42 makes the random sampling reproducible.
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

# 3. Run the random search
random_search.fit(X_train, y_train)

# 4. Report the winner
print(f"\nBest Parameters found by Random Search: {random_search.best_params_}")
print(f"Best cross-validated score: {random_search.best_score_:.2%}")

# Test-set evaluation for fair comparison with baseline and grid search
y_pred_random = random_search.predict(X_test)
accuracy_random = accuracy_score(y_test, y_pred_random)
print(f"Test-set accuracy of the tuned model: {accuracy_random:.2%}")

# The lecture (Slide 26) recommends keeping a compact experiment log.
# Here is a minimal one comparing all three approaches so far:
print("\n--- Experiment log (per Slide 26: 'Log Everything') ---")
print(f"  Baseline (defaults):   test acc = {accuracy_baseline:.2%}")
print(f"  Grid Search:           CV = {grid_search.best_score_:.2%},  test acc = {accuracy_grid:.2%}")
print(f"  Random Search:         CV = {random_search.best_score_:.2%},  test acc = {accuracy_random:.2%}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best Parameters found by Random Search: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 20}
Best cross-validated score: 94.29%
Test-set accuracy of the tuned model: 100.00%

--- Experiment log (per Slide 26: 'Log Everything') ---
  Baseline (defaults):   test acc = 100.00%
  Grid Search:           CV = 94.29%,  test acc = 100.00%
  Random Search:         CV = 94.29%,  test acc = 100.00%


## Part 5: Introduction to AutoML with AutoGluon

**Concept:** AutoML takes hyperparameter tuning to the next level. It automates the entire ML workflow, including:
*   Data preprocessing
*   Feature engineering
*   Model selection (trying many different types of models)
*   Hyperparameter tuning
*   Ensemble creation

**AutoGluon** is a popular and easy-to-use AutoML library. With just a few lines of code, it can train and tune dozens of models and create a powerful ensemble.

**This part is fully coded.** Your task is to run it and see the power of AutoML. Note that it may take a few minutes to run.

In [6]:
# You may need to install AutoGluon first. Uncomment the line below in your Colab notebook.
!pip install autogluon

from autogluon.tabular import TabularPredictor

# AutoGluon expects ONE DataFrame containing both features AND the target column,
# rather than separate X and y arrays like sklearn. This is a small API difference
# but worth knowing -- the underlying workflow is the same.
train_data_ag = pd.DataFrame(X_train, columns=iris.feature_names)
train_data_ag['species'] = y_train

# Build the matching test DataFrame too
test_data_ag = pd.DataFrame(X_test, columns=iris.feature_names)
test_data_ag['species'] = y_test

# Build and fit the AutoGluon predictor.
# The 4-stage automated pipeline from the lecture (Slide 15) runs internally:
#   1) Preprocessing (impute / encode / scale)
#   2) Model Family selection (trees, linear, boosting, neural nets, etc.)
#   3) Hyperparameter optimization within each candidate
#   4) Ensemble of the strongest models  <-- this is the AutoML 'extra' over manual tuning
# time_limit=60 caps the search at 60 seconds, addressing the runtime constraint from Slide 6.
predictor = TabularPredictor(label='species', eval_metric='accuracy').fit(
    train_data=train_data_ag, time_limit=60
)

# The leaderboard is the key output (Slide 23): a ranking of every model AutoGluon trained,
# scored on both validation and the held-out test data.
leaderboard = predictor.leaderboard(test_data_ag)
print(leaderboard)

No path specified. Models will be saved in: "AutogluonModels/ag-20260506_200110"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.80 GB / 12.67 GB (85.3%)
Disk Space Avail:   75.27 GB / 107.72 GB (69.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.

                  model  score_test  score_val eval_metric  pred_time_test  \
0               XGBoost    1.000000   0.952381    accuracy        0.025023   
1      RandomForestGini    1.000000   0.857143    accuracy        0.092117   
2      RandomForestEntr    1.000000   0.857143    accuracy        0.097722   
3        ExtraTreesEntr    1.000000   0.904762    accuracy        0.098605   
4        ExtraTreesGini    1.000000   0.904762    accuracy        0.098857   
5         LightGBMLarge    0.977778   0.857143    accuracy        0.001514   
6              LightGBM    0.955556   0.857143    accuracy        0.001093   
7        NeuralNetTorch    0.955556   0.952381    accuracy        0.009635   
8            LightGBMXT    0.933333   0.952381    accuracy        0.001478   
9              CatBoost    0.933333   0.952381    accuracy        0.003323   
10  WeightedEnsemble_L2    0.933333   0.952381    accuracy        0.003513   
11      NeuralNetFastAI    0.866667   0.904762    accuracy      

## 📝 Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

1.  **What is the main difference between a model parameter and a hyperparameter?**
2.  **When would you choose to use Grid Search over Random Search, and vice-versa?**
3.  **Looking at the AutoGluon leaderboard, which model performed the best? What does AutoML do that makes it so powerful compared to manual tuning?**

---

### My Answers

**1. The difference between a model parameter and a hyperparameter:**

The simplest way to keep them straight is the lecture's **recipe analogy** (Slide 4):

- **Hyperparameters** are like the **oven temperature, cooking time, and amount of seasoning** — you decide them *before* the cake bakes. Examples: `max_depth`, `learning_rate`, `n_estimators`, the `C` regularization strength. They don't appear in the data; they're choices the analyst makes about *how* the algorithm should learn.
- **Model parameters** are the **final taste of the dish** — the values the algorithm computes from the data while it's training. Examples: the coefficients in a logistic regression, the split thresholds inside a tree, the weights inside a neural network. The model figures these out on its own; we don't set them.

The reason the distinction matters: **hyperparameters shape how the parameters are learned.** They control model capacity (how complex it can get), regularization (how much overfitting is penalized), and search behavior (how the optimizer moves through parameter space). Tuning hyperparameters changes the *learning process itself*, not just the fitted values. A different oven temperature gives you a completely different cake even from the same batter.

**2. When to choose Grid Search vs Random Search:**

Both are systematic ways of searching the hyperparameter space, but they make different tradeoffs (Slide 9 vs Slide 10).

**Use Grid Search when:**
- The search space is **small** — maybe 2 or 3 hyperparameters with a handful of candidate values each. Our Task 1 grid had 3 × 3 = 9 combinations, which is fine.
- You want **transparency and reproducibility** — you can list exactly which configurations were tried and explain the result to a stakeholder.
- You already have **strong domain knowledge** about which range of values is worth inspecting carefully, and you want to be exhaustive within that narrow range.
- Each individual model fit is **cheap** so the combinatorial blowup doesn't matter.

**Use Random Search when:**
- The search space is **larger** with more hyperparameters or wider ranges. Our Task 2 space had 10 × 5 × 3 = 150 possible combinations; an exhaustive grid would be 17× more expensive than the 10-iteration random search we actually ran.
- You don't yet know which hyperparameters matter most. The lecture's key insight (Slide 10) is that **"many hyperparameters matter little, while a few dominate"** — random sampling spreads trials across more unique values for the dominant dimensions, so it tends to reach good regions faster than a grid.
- You have a **fixed compute budget** and want the most exploration possible per dollar. With grid search you're stuck with whatever combination count your grid produces; with random search you set `n_iter` directly to your budget.
- The lecture's **restaurant-in-a-city analogy** (Slide 10): visiting every block is systematic, but sampling across neighborhoods finds promising districts much faster.

**A practical pattern** (Slide 13): start with **Random Search** as a fast first baseline to find roughly which region is promising, then switch to a focused **Grid Search** in that narrowed region for the final fine-tune. And once each evaluation gets very expensive (e.g., gradient-boosted models with heavy preprocessing), graduate to **Bayesian Optimization** (Slide 11) so the search learns from past trials instead of treating each one independently — the lecture compares it to a hot-and-cold game where each guess teaches you something about where the better answers are.

**3. AutoGluon leaderboard winner and what AutoML adds:**

On my run, **multiple models tied at 100% test accuracy** — RandomForest (Gini and Entropy variants), ExtraTrees (Gini and Entropy variants), and the **`WeightedEnsemble_L2`** all hit a perfect score on the held-out test set. Looking at the *validation* score breaks the tie: the ExtraTrees variants and the WeightedEnsemble_L2 led at ~0.905, with the RandomForest variants slightly behind at ~0.857. AutoGluon designated the **`WeightedEnsemble_L2`** as its "Best model" — this is the model AutoGluon *constructs* by taking a weighted combination of the strongest individual learners, which is the AutoML extra over plain hyperparameter tuning. (Worth knowing: in environments with the full dependency set installed — LightGBM, XGBoost, CatBoost, and neural networks — the leaderboard would include those models too and the comparison would be more interesting.)

What AutoML does that manual tuning doesn't — the **automated pipeline stack from Slide 15**:

1. **Preprocessing automation** — imputation, encoding, scaling are all chosen automatically. We didn't have to think about whether to standardize.
2. **Model family selection** — instead of committing to Random Forest from the start (which is what we did manually), AutoGluon compared Random Forest, Extra Trees, and would have tried LightGBM, XGBoost, CatBoost, and neural networks if those packages were installed. Manual tuning of one family can't catch the case where a *different* family is fundamentally better suited to the data.
3. **Hyperparameter optimization within each candidate** — it tunes each candidate algorithm internally, so we're not comparing a tuned RF against an untuned XGBoost. The comparison is fair.
4. **Ensemble construction** — this is the piece that goes beyond what Grid/Random Search can do. AutoGluon doesn't just pick the single best model; it builds a *weighted ensemble* of the top performers (the `WeightedEnsemble_L2` row), often squeezing out an extra fraction of a percent that no individual model achieved.
5. **Leaderboard ranking** — we get to *see* the comparison instead of having to design it ourselves.

All of this in the time it took to type four lines of code, where the manual workflow above took us multiple cells of careful Grid/Random search setup just for one algorithm.

**An important caveat from the deck (Slide 22):** Iris is exactly the kind of "tiny dataset" where AutoML can search aggressively enough to overfit the *validation process itself*. Five different models all hitting 100% test accuracy doesn't mean they're equally good — it means our test set is too small to differentiate them. On a real-world tabular dataset with thousands of rows, the AutoML advantage would be much more visible, and we'd actually be able to tell which models work best.

## 🤔 Reflection

**What I learned:**

The biggest mental shift this module was treating tuning as a **controlled experiment, not a sequence of lucky guesses** (Slide 26). Before this lab, I would have just tried a few `n_estimators` values by hand and called it good. Now I see why the discipline matters: every trial needs the same rules (cross-validation as the **referee**, Slide 5), the same metric (chosen up front before the search begins, Slide 6), and the same logbook (configurations, scores, runtimes, observations). Without those guardrails, you don't actually know whether your "improvement" is real or just noise from a lucky split.

The lecture's **"learning arc"** on Slide 2 gave me a clean mental map I didn't have before: **Defaults → Validation → Search → AutoML → Decision Rule**. Each stage exists because the previous one isn't sufficient on its own:
- Defaults are convenient but generic (the **shoes analogy** — Slide 3 — they walk safely but don't fit your foot).
- Validation comes next because a single split can be misleading.
- Search comes next because manual one-at-a-time tuning doesn't scale.
- AutoML comes next because real projects involve preprocessing + model choice + hyperparameters all interacting (Slide 14's "decision explosion").
- The decision rule comes last because automation never replaces human judgment about whether the result is actually deployable.

**Challenges:**

The trickiest concept was understanding the **exploration vs exploitation tension** that drives Bayesian Optimization (Slide 11). My first instinct was "just keep going to the best region we've seen so far," but that's pure exploitation — you'd miss potentially better regions that haven't been sampled yet. The hot-and-cold game analogy made it click: each guess teaches you something, but you still need to occasionally test surprising places to make sure there isn't a better answer hiding out there. This is the mathematical structure behind why Bayesian Optimization is more sample-efficient than Random Search on expensive evaluations.

Another challenge was accepting that on this lab's specific data, the tuning didn't visibly help. The baseline, Grid Search, and Random Search all produced 100% test accuracy. That feels anticlimactic, but it matches the deck's Slide 22 warning about **tiny datasets** — Iris simply doesn't have enough complexity to differentiate between a default Random Forest and a tuned one. The methodology lesson is what mattered, not the score.

**Connections:**

This module ties almost everything we've done into one workflow:
- **Module 07 (model evaluation)**: cross-validation here is exactly the referee Slide 5 describes — same idea, now applied as the inner loop of every tuning trial.
- **Module 08 (bias-variance)**: tuning is the practical knob for moving along the bias-variance curve. `max_depth=5` biases the forest toward simplicity (less variance); `max_depth=None` lets it overfit (more variance). The grid is essentially exploring different points on that curve.
- **Module 09 (ensembles)**: AutoGluon's `WeightedEnsemble_L2` is the ensembling principle from Slide 18 of last module — the wisdom of the crowd applied across model *families*, not just trees within one forest.
- **Module 10 (unsupervised)**: PCA-then-K-Means and AutoML are both "pipeline" thinking — a sequence of steps that has to be tuned together, not independently.

**Takeaways for future projects:**

1. **Always start with a baseline** before reaching for tuning. The lecture's logic: you can't claim improvement without knowing what you're improving over. The default-config Random Forest in Part 2 was that anchor for everything else.
2. **Choose the metric before the search begins** (Slide 6's funnel: Goal → Metric → Constraints → Search). The wrong rubric will optimize the wrong behavior efficiently. Decide what matters — accuracy, F1, recall, RMSE, latency — first.
3. **Random Search before Grid Search.** The pragmatic pattern from Slide 13: random sampling first to find the promising region, then a tighter grid (or Bayesian optimization) to fine-tune within that region. Going straight to grid search blows your compute budget on the wrong area.
4. **Use AutoML as a fast baseline, not a substitute for thinking.** Slide 18's positioning: "Faster baseline, not blind trust." AutoGluon gives me a strong starting point in minutes, but I still need to validate on holdout data (Slide 24), watch for the tiny-dataset trap (Slide 22), and keep a simpler interpretable model alongside it for audit and debugging (Slide 25's "reference model").
5. **Keep humans in the loop on the bookends, not the middle** (Slide 16, 17). Setup (problem definition, metric, constraints) and Selection (which model actually goes to production, what tradeoffs we accepted) stay human. The middle — evaluating thousands of configurations — is what we hand off to the machine.
6. **Avoid AutoML when transparency is non-negotiable** (Slide 22): healthcare, finance, hiring, or anywhere a regulator can demand to know exactly *why* a decision was made. A slightly weaker but interpretable model can be the safer engineering choice.